# Model Predictions (qubit-TransmonCross-Hamiltonian_params)
## Hamiltonian to Quantum Metal

Inverse model with surrogate-defined loss

## Configuration

In [19]:
## the parameter file has the hyperparameters
## start there if you want to change the setup

from parameters_surrogate_defined_loss import *

## Library

In [20]:
import os, gc
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras.models import load_model

In [21]:
import time
import platform
import json

## Dataset

### Load

In [22]:
## load the held-out arrays saved from ml_00
X_test = np.load(f'{DATA_DIR}/npy/x_test_one_hot_encoding_augmented.npy', allow_pickle=True)
y_test = np.load(f'{DATA_DIR}/npy/y_test_one_hot_encoding_augmented.npy', allow_pickle=True)

with open(str(Path(METADATA_DIR) / 'X_names'), 'r') as f:
    Hamiltonian_column_names = f.read().splitlines()

qiskit_param_names = np.load(str(Path(METADATA_DIR) / 'y_columns.npy'), allow_pickle=True).astype(str).tolist()

print(f'Inputs (Hamiltonian):     {X_test.shape[1]} columns')
print(f'Outputs (Quantum Metal):  {y_test.shape[1]} columns')
print(f'Test samples: {len(X_test)}')
print(f'Hamiltonian columns: {Hamiltonian_column_names}')
print(f'Quantum Metal columns: {qiskit_param_names}')

Inputs (Hamiltonian):     1 columns
Outputs (Quantum Metal):  3 columns
Test samples: 291
Hamiltonian columns: ['EC_GHz']
Quantum Metal columns: ['design_options.connection_pads.readout.claw_length', 'design_options.connection_pads.readout.ground_spacing', 'design_options.cross_length']


### Define conversion layer

In [23]:
## must define this class before loading the saved combined model
class ScalerConversionLayer(tf.keras.layers.Layer):
    def __init__(self, scale_a, scale_b, **kwargs):
        kwargs.setdefault('trainable', False)
        super().__init__(**kwargs)
        self._scale_a = tf.constant(scale_a, dtype=tf.float32)
        self._scale_b = tf.constant(scale_b, dtype=tf.float32)
        self._cfg = dict(
            scale_a=list(scale_a) if hasattr(scale_a, '__iter__') else scale_a,
            scale_b=list(scale_b) if hasattr(scale_b, '__iter__') else scale_b)

    def call(self, inputs):
        a = tf.cast(self._scale_a, inputs.dtype)
        b = tf.cast(self._scale_b, inputs.dtype)
        return inputs * a + b

    def get_config(self):
        config = super().get_config()
        config.update(self._cfg)
        return config

### Load model and predict

In [24]:
## load the saved combined inverse plus surrogate model
encoding = 'surrogate_defined_loss'
chosen_path = f'{MODEL_DIR}/best_keras_model_{encoding}.keras'
X_test_cur = np.asarray(X_test)
y_test_cur = np.asarray(y_test)
headers = Hamiltonian_column_names
print(f'Model path: {chosen_path}')

Model path: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/best_keras_model_surrogate_defined_loss.keras


In [25]:
## run the model on the held-out test set
tf.keras.backend.clear_session()
gc.collect()
try:
    tf.config.experimental.reset_memory_stats('GPU:0')
except Exception:
    pass

with tf.device('/CPU:0'):
    combined_model = load_model(
        chosen_path,
        compile=False,
        custom_objects={'ScalerConversionLayer': ScalerConversionLayer},
    )
    inverse_model = combined_model.get_layer('inverse_model')
    predictions = combined_model.predict(X_test_cur, verbose=0)

if isinstance(predictions, list):
    Hamiltonian_pred = predictions[0]
    qiskit_pred = predictions[1]
else:
    Hamiltonian_pred = predictions
    qiskit_pred = inverse_model.predict(X_test_cur, verbose=0)

print(f'Samples: {len(X_test_cur)}')
print(f'Hamiltonian target dim: {X_test_cur.shape[1]}')
print(f'Quantum Metal output dim: {y_test_cur.shape[1]}')

Samples: 291
Hamiltonian target dim: 1
Quantum Metal output dim: 3


In [26]:
## runtime benchmark for the paper speedup claim.
## measures persample CPU inference time for
## (1) the inverse MLP alone (hamiltonian > qiskit params)
## (2) the combined pipeline (hamiltonian > qiskit params > reconstructed hamiltonian)
## reports singlesample and batch timings so the paper can quote the
## endtoend design generation cost on a standard CPU.

def _bench(predict_fn, X, n_warmup=10, n_repeat=50, single_sample=False):
    """Time a keras .predict() call. Returns (mean_ms, std_ms) per call."""
    ## warmup to trigger tf.function tracing, XLA compilation, etc.
    for _ in range(n_warmup):
        _ = predict_fn(X[:1] if single_sample else X, verbose=0)
    times = []
    for _ in range(n_repeat):
        t0 = time.perf_counter()
        _ = predict_fn(X[:1] if single_sample else X, verbose=0)
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000.0)  ## ms
    return float(np.mean(times)), float(np.std(times))

X_bench = np.asarray(X_test_cur, dtype=np.float32)
N_bench = len(X_bench)

with tf.device('/CPU:0'):
    ## inverse model alone
    inv_batch_mean_ms, inv_batch_std_ms = _bench(inverse_model.predict, X_bench, single_sample=False)
    inv_single_mean_ms, inv_single_std_ms = _bench(inverse_model.predict, X_bench, single_sample=True)

    ## combined pipeline (inverse + scaler conversion + frozen surrogate)
    comb_batch_mean_ms, comb_batch_std_ms = _bench(combined_model.predict, X_bench, single_sample=False)
    comb_single_mean_ms, comb_single_std_ms = _bench(combined_model.predict, X_bench, single_sample=True)

## persample numbers for batched inference
inv_per_sample_ms = inv_batch_mean_ms / N_bench
comb_per_sample_ms = comb_batch_mean_ms / N_bench

## speedup vs ansys (2 min per Q3D extraction, measured separately)
ansys_min_per_sample = 2.0
ansys_ms_per_sample = ansys_min_per_sample * 60.0 * 1000.0
speedup_inv_vs_ansys = ansys_ms_per_sample / inv_per_sample_ms
speedup_comb_vs_ansys = ansys_ms_per_sample / comb_per_sample_ms
speedup_single_vs_ansys = ansys_ms_per_sample / comb_single_mean_ms

In [27]:
## pretty print
print('=' * 78)
print(' Runtime benchmark on CPU (paper Section "Runtime and speedup")')
print('=' * 78)
print(f' Hardware : {platform.processor() or platform.machine()}')
print(f' System   : {platform.system()} {platform.release()}')
print(f' Python   : {platform.python_version()}')
print(f' TF       : {tf.__version__}')
print(f' N_test   : {N_bench} samples')
print(f' Warmup   : 10 calls | Repeat: 50 calls')
print('-' * 78)
print(f' Inverse MLP (Hamiltonian -> Quantum Metal params)')
print(f'   batch  : {inv_batch_mean_ms:8.3f} +/- {inv_batch_std_ms:.3f} ms total  '
      f'({inv_per_sample_ms * 1000.0:7.2f} us / sample)')
print(f'   single : {inv_single_mean_ms:8.3f} +/- {inv_single_std_ms:.3f} ms / call')
print('-' * 78)
print(f' Combined pipeline (Hamiltonian -> Quantum Metal -> reconstructed Hamiltonian)')
print(f'   batch  : {comb_batch_mean_ms:8.3f} +/- {comb_batch_std_ms:.3f} ms total  '
      f'({comb_per_sample_ms * 1000.0:7.2f} us / sample)')
print(f'   single : {comb_single_mean_ms:8.3f} +/- {comb_single_std_ms:.3f} ms / call')
print('-' * 78)
print(f' Ansys Q3D reference  : ~{ansys_min_per_sample:.0f} min / sample '
      f'(~{ansys_ms_per_sample:.2e} ms / sample)')
print(f' Speedup (inverse MLP batch, per-sample) : {speedup_inv_vs_ansys:.2e}  '
      f'({int(np.log10(speedup_inv_vs_ansys))} orders of magnitude)')
print(f' Speedup (combined batch, per-sample)    : {speedup_comb_vs_ansys:.2e}  '
      f'({int(np.log10(speedup_comb_vs_ansys))} orders of magnitude)')
print(f' Speedup (combined single-sample call)   : {speedup_single_vs_ansys:.2e}  '
      f'({int(np.log10(speedup_single_vs_ansys))} orders of magnitude)')
print('=' * 78)

## save for later
runtime_stats = {
    'hardware': platform.processor() or platform.machine(),
    'system': f'{platform.system()} {platform.release()}',
    'python': platform.python_version(),
    'tf': tf.__version__,
    'n_test': N_bench,
    'inverse_batch_total_ms': inv_batch_mean_ms,
    'inverse_per_sample_ms': inv_per_sample_ms,
    'inverse_single_call_ms': inv_single_mean_ms,
    'combined_batch_total_ms': comb_batch_mean_ms,
    'combined_per_sample_ms': comb_per_sample_ms,
    'combined_single_call_ms': comb_single_mean_ms,
    'ansys_min_per_sample': ansys_min_per_sample,
    'speedup_combined_per_sample': speedup_comb_vs_ansys,
    'speedup_combined_single_call': speedup_single_vs_ansys,
}

runtime_path = Path(RESULTS_DIR) / 'runtime' / 'runtime_benchmark.json'
runtime_path.parent.mkdir(parents=True, exist_ok=True)
with runtime_path.open('w') as f:
    json.dump(runtime_stats, f, indent=2)
print(f'\nSaved runtime benchmark to {runtime_path}')

 Runtime benchmark on CPU (paper Section "Runtime and speedup")
 Hardware : x86_64
 System   : Linux 6.3.12-200.fc38.x86_64
 Python   : 3.10.13
 TF       : 2.20.0
 N_test   : 291 samples
 Warmup   : 10 calls | Repeat: 50 calls
------------------------------------------------------------------------------
 Inverse MLP (Hamiltonian -> Quantum Metal params)
   batch  :  117.754 +/- 9.979 ms total  ( 404.65 us / sample)
   single :  104.310 +/- 9.785 ms / call
------------------------------------------------------------------------------
 Combined pipeline (Hamiltonian -> Quantum Metal -> reconstructed Hamiltonian)
   batch  :  120.487 +/- 12.703 ms total  ( 414.04 us / sample)
   single :  102.948 +/- 11.906 ms / call
------------------------------------------------------------------------------
 Ansys Q3D reference  : ~2 min / sample (~1.20e+05 ms / sample)
 Speedup (inverse MLP batch, per-sample) : 2.97e+05  (5 orders of magnitude)
 Speedup (combined batch, per-sample)    : 2.90e+05  (5

## Hamiltonian Reconstruction Errors

In [28]:
## unscale the target and reconstructed Hamiltonian values
with open(str(Path(METADATA_DIR) / 'X_names'), 'r') as f:
    Hamiltonian_names = f.read().splitlines()
qiskit_names = np.load(str(Path(METADATA_DIR) / 'y_columns.npy'), allow_pickle=True).astype(str).tolist()

X_test_cur = np.asarray(X_test_cur, dtype=np.float32)
y_test_cur = np.asarray(y_test_cur, dtype=np.float32)
Hamiltonian_pred = np.asarray(Hamiltonian_pred, dtype=np.float32)
qiskit_pred = np.asarray(qiskit_pred, dtype=np.float32)

X_test_unscaled = X_test_cur.copy()
Hamiltonian_pred_unscaled = Hamiltonian_pred.copy()
for j, Hamiltonian_name in enumerate(Hamiltonian_names[:X_test_cur.shape[1]]):
    scaler = joblib.load(f'{SCALERS_DIR}/scaler_X_{Hamiltonian_name}.save')
    X_test_unscaled[:, j] = scaler.inverse_transform(X_test_cur[:, [j]]).ravel()
    Hamiltonian_pred_unscaled[:, j] = scaler.inverse_transform(Hamiltonian_pred[:, [j]]).ravel()

qiskit_pred_unscaled = qiskit_pred.copy()
y_test_unscaled = y_test_cur.copy()
for j, col_name in enumerate(qiskit_names[:qiskit_pred.shape[1]]):
    scaler = joblib.load(f'{SCALERS_DIR}/scaler_y_{col_name}_one_hot_encoding.save')
    qiskit_pred_unscaled[:, j] = scaler.inverse_transform(qiskit_pred[:, [j]]).ravel()
    y_test_unscaled[:, j] = scaler.inverse_transform(y_test_cur[:, [j]]).ravel()

prediction_rows = []
for i in range(X_test_unscaled.shape[0]):
    row = {'sample_idx': i}
    for j, name in enumerate(Hamiltonian_names[:X_test_unscaled.shape[1]]):
        row[f'target_{name}'] = X_test_unscaled[i, j]
        row[f'pred_{name}'] = Hamiltonian_pred_unscaled[i, j]
        row[f'abs_error_{name}'] = abs(Hamiltonian_pred_unscaled[i, j] - X_test_unscaled[i, j])
    for j, name in enumerate(qiskit_names[:qiskit_pred_unscaled.shape[1]]):
        short = name.replace('design_options.', '')
        row[f'ref_{short}'] = y_test_unscaled[i, j]
        row[f'pred_{short}'] = qiskit_pred_unscaled[i, j]
    prediction_rows.append(row)

pred_df = pd.DataFrame(prediction_rows)
predictions_path = Path(RESULTS_DIR) / 'predictions' / f'surrogate_loss_Hamiltonian_reconstruction_unscaled_{encoding}.csv'
predictions_path.parent.mkdir(parents=True, exist_ok=True)
pred_df.to_csv(predictions_path, index=False, float_format='%.8g')
print(f'Saved unscaled predictions to {predictions_path}')

Saved unscaled predictions to /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/results/predictions/surrogate_loss_Hamiltonian_reconstruction_unscaled_surrogate_defined_loss.csv


In [29]:
## save the percent-error table used by figures/paper/build_manuscript_exports.py
eps = 1e-15
pct_errors_unscaled = 100.0 * np.abs(Hamiltonian_pred_unscaled - X_test_unscaled) / (np.abs(X_test_unscaled) + eps)

percent_error_df = pd.DataFrame({
    'EC': pct_errors_unscaled[:, 0],
})
percent_error_path = Path(RESULTS_DIR) / 'validation' / 'inverse+surrogate_percentErrors.csv'
percent_error_path.parent.mkdir(parents=True, exist_ok=True)
percent_error_df.to_csv(percent_error_path, index=False)
print(f'Saved manuscript percent errors to {percent_error_path}')

summary = percent_error_df.agg(['mean', 'median', 'std', 'min', 'max']).T
print(summary.round(4))

Saved manuscript percent errors to /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/results/validation/inverse+surrogate_percentErrors.csv
      mean  median     std     min     max
EC  0.0646  0.0141  0.1083  0.0002  0.6853
